In [1]:
import numpy as np

# ── 1. Define the Network Structure & CPTs ────────────────────────────────────
# Problem: Predict if a person is Late to Work
# Dependencies: Rain causes Traffic & Accident; both cause being Late

# P(Rain)
P_rain = {'yes': 0.3, 'no': 0.7}

# P(Traffic | Rain)  — Rain increases traffic
P_traffic = {
    'yes': {'yes': 0.8, 'no': 0.2},   # Rain=yes → Traffic:80%
    'no':  {'yes': 0.1, 'no': 0.9},   # Rain=no  → Traffic:10%
}

# P(Accident | Rain)  — Rain increases accident risk
P_accident = {
    'yes': {'yes': 0.4, 'no': 0.6},   # Rain=yes → Accident:40%
    'no':  {'yes': 0.05,'no': 0.95},  # Rain=no  → Accident:5%
}

# P(Late | Traffic, Accident)  — Conditional Probability Table
P_late = {
    ('yes','yes'): {'yes': 0.95, 'no': 0.05},  # Traffic+Accident → almost always late
    ('yes','no'):  {'yes': 0.70, 'no': 0.30},  # Traffic only → 70% late
    ('no', 'yes'): {'yes': 0.60, 'no': 0.40},  # Accident only → 60% late
    ('no', 'no'):  {'yes': 0.05, 'no': 0.95},  # Neither → rarely late
}

# ── 2. Inference by Enumeration ───────────────────────────────────────────────
# Compute P(Late=yes) by summing over all hidden variable combinations
def predict_late():
    p_late_yes = 0.0

    for rain in ['yes', 'no']:
        for traffic in ['yes', 'no']:
            for accident in ['yes', 'no']:

                # Joint probability of this combination
                joint = (P_rain[rain]
                       * P_traffic[rain][traffic]
                       * P_accident[rain][accident]
                       * P_late[(traffic, accident)]['yes'])

                p_late_yes += joint

    return p_late_yes

# ── 3. Conditional Inference — Given Evidence ─────────────────────────────────
# P(Late=yes | Rain=yes) — it IS raining, what's the chance of being late?
def predict_late_given_rain(observed_rain):
    p_late_yes = 0.0

    for traffic in ['yes', 'no']:
        for accident in ['yes', 'no']:
            joint = (P_traffic[observed_rain][traffic]
                   * P_accident[observed_rain][accident]
                   * P_late[(traffic, accident)]['yes'])
            p_late_yes += joint

    return p_late_yes

# ── 4. Results ─────────────────────────────────────────────────────────────────
p_late          = predict_late()
p_late_rain     = predict_late_given_rain('yes')
p_late_no_rain  = predict_late_given_rain('no')

print(f"P(Late)            : {p_late:.4f}  ({p_late*100:.1f}%)")
print(f"P(Late | Rain=Yes) : {p_late_rain:.4f}  ({p_late_rain*100:.1f}%)")
print(f"P(Late | Rain=No)  : {p_late_no_rain:.4f}  ({p_late_no_rain*100:.1f}%)")
print()
print("Conclusion:", "Rain significantly increases the probability of being Late!"
      if p_late_rain > 2*p_late_no_rain else "Rain has mild effect.")

P(Late)            : 0.3069  (30.7%)
P(Late | Rain=Yes) : 0.6940  (69.4%)
P(Late | Rain=No)  : 0.1410  (14.1%)

Conclusion: Rain significantly increases the probability of being Late!
